# AutoNex: about 700K vs about 500M parameters

This Colab notebook builds comparison text directly from the supplied vehicle CSV, makes a make/model-grouped split, and trains two decoder-only character models from random initialization. The smaller configuration is sized near the existing AutoNex model. The larger configuration is approximately 500 million parameters; its exact count is computed from the instantiated architecture.

The structured CSV is not a language corpus. The notebook prints the number of unique make/model groups and the exact comparison examples it constructs. Each pair is made once per supported priority inside one split. It does not duplicate examples to inflate the counts.

The 500M model may overfit or fail to learn useful language from this small, template-generated corpus. The full 500M run is off by default. Colab checks the current hardware, estimates memory, and runs a short large-model smoke step only when the estimate leaves headroom. The smoke run is not a completed training run. A full large-model run requires a separate opt-in after reviewing the measured speed and estimated duration.

Model-written text is evaluated with held-out next-character loss/perplexity and a numeric-fact screen, then printed for human review. Loss and perplexity do not establish factuality or readability. No conventional accuracy is reported.


## 1. Runtime and artifact folder

Use a Colab runtime with a GPU for the large-model smoke check. The notebook can train the smaller model on CPU, though it will be slower. For resumable checkpoints, mount Google Drive or download the small checkpoint after training. The 500M resumable file can be several gigabytes and should be saved to Drive.

The image-identification model used by the website is separate. This notebook trains only the vehicle-comparison text models.


In [ ]:
import json
import hashlib
import math
import os
import random
import re
import string
import time
from itertools import combinations
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

try:
    import psutil
except ImportError:
    psutil = None

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IS_COLAB = False
try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    files = None

SAVE_TO_DRIVE = True  # Set False to keep outputs in the temporary Colab runtime.
if IS_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/AutoNex_Comparison_Artifacts")
else:
    ARTIFACT_DIR = Path("/content/AutoNex_Comparison_Artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", os.sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if psutil is not None:
    memory = psutil.virtual_memory()
    print(f"Host RAM total/free: {memory.total / 2**30:.2f} / {memory.available / 2**30:.2f} GiB")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU memory free/total: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB")
else:
    print("No GPU is active. The 500M training cells will remain skipped.")


## 2. Upload or select the vehicle dataset

By default, the next cell opens the Colab upload picker. To use a Drive copy instead, set DATA_SOURCE to drive and edit DRIVE_CSV_PATH. The notebook requires Make and Model; other columns are optional and only contribute facts when present.


In [ ]:
DATA_SOURCE = "upload"  # Change to "drive" to read a CSV from mounted Google Drive.
DRIVE_CSV_PATH = "/content/drive/MyDrive/cars_dataset_100000_india_global.csv"

if DATA_SOURCE == "upload":
    if not IS_COLAB:
        raise RuntimeError("Upload mode requires Google Colab. Set DATA_SOURCE='drive' in Colab.")
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("Upload a CSV vehicle dataset.")
    DATA_PATH = Path("/content") / csv_files[0]
elif DATA_SOURCE == "drive":
    if not IS_COLAB:
        raise RuntimeError("Drive mode requires Google Colab.")
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DATA_PATH = Path(DRIVE_CSV_PATH)
else:
    raise ValueError("DATA_SOURCE must be 'upload' or 'drive'.")

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset CSV not found: {DATA_PATH}")
print("Selected dataset:", DATA_PATH)


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = [str(column).strip() for column in df.columns]

required_columns = {"Make", "Model"}
missing_required = sorted(required_columns - set(df.columns))
if missing_required:
    raise ValueError(
        f"Dataset is missing required columns: {missing_required}. "
        f"Available columns: {df.columns.tolist()}"
    )

optional_columns = [
    "Trim_Variant", "Year", "Body_Style", "Doors", "Seating_Capacity",
    "Drivetrain", "Transmission", "Fuel_Type", "Displacement", "Cylinders",
    "Power_BHP", "Mileage_City", "Mileage_Highway",
    "Formatted_Price", "Ex_Showroom_Price_INR",
]
print("CSV rows:", len(df))
print("CSV columns:", len(df.columns))
print("Available comparison fields:", [name for name in optional_columns if name in df.columns])
print("Optional fields absent:", [name for name in optional_columns if name not in df.columns])

def clean_value(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip()
    if text.casefold() in {"", "nan", "none", "null"}:
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return text

def normalized_group(value):
    text = clean_value(value).casefold()
    return " ".join(re.sub(r"[^\w]+", " ", text, flags=re.UNICODE).split())

fact_columns = [
    "Year", "Trim_Variant", "Body_Style", "Doors", "Seating_Capacity",
    "Drivetrain", "Transmission", "Fuel_Type", "Displacement", "Cylinders",
    "Power_BHP", "Mileage_City", "Mileage_Highway",
    "Formatted_Price", "Ex_Showroom_Price_INR",
]
available_fact_columns = [name for name in fact_columns if name in df.columns]


## 3. Build grounded examples and split by make/model

A representative record is selected from each normalized Make + Model group. When a model has several rows, the chosen row is the one with the most available comparison fields, with newer year used as a deterministic tie-breaker. The full model group is assigned to exactly one split before any comparison pairs are generated.

Pairs are created only among groups in the same split. This is a strict unseen-model evaluation: validation and test model names do not appear in training prompts. It creates fewer examples than a row-random split, which is the intended trade-off for measuring generalization to unseen models.


In [ ]:
work = df.copy()
work["_make_norm"] = work["Make"].map(normalized_group)
work["_model_norm"] = work["Model"].map(normalized_group)
work["_group_id"] = work["_make_norm"] + " | " + work["_model_norm"]
work = work[(work["_make_norm"] != "") & (work["_model_norm"] != "")].copy()

def row_value(row, column):
    return clean_value(row.get(column, ""))

def row_price(row):
    formatted = row_value(row, "Formatted_Price")
    if formatted:
        return formatted
    raw = row_value(row, "Ex_Showroom_Price_INR")
    return f"{raw} INR" if raw else ""

score_columns = [name for name in available_fact_columns if name not in {"Formatted_Price"}]
work["_completeness"] = (
    work[score_columns].apply(lambda column: column.map(clean_value).ne("")).sum(axis=1)
    if score_columns else 0
)
if "Year" in work.columns:
    work["_year_order"] = pd.to_numeric(work["Year"], errors="coerce").fillna(0)
else:
    work["_year_order"] = 0
work["_variant_order"] = work["Trim_Variant"].map(clean_value) if "Trim_Variant" in work.columns else ""

representatives = (
    work.sort_values(
        ["_group_id", "_completeness", "_year_order", "_variant_order"],
        ascending=[True, False, False, True],
        kind="mergesort",
    )
    .drop_duplicates("_group_id", keep="first")
)

def make_record(row):
    make = row_value(row, "Make")
    model = row_value(row, "Model")
    variant = row_value(row, "Trim_Variant")
    year = row_value(row, "Year")
    name = " ".join(part for part in [make, model] if part)
    if variant:
        name += f" ({variant})"
    if year:
        name += f" · {year}"
    values = {column: row_value(row, column) for column in available_fact_columns}
    values["Formatted_Price"] = row_price(row)
    return {
        "group_id": row["_group_id"],
        "make": make,
        "model": model,
        "name": name,
        "values": values,
    }

records = [make_record(row) for _, row in representatives.iterrows()]
if len(records) < 3:
    raise ValueError("At least three distinct make/model groups are needed for train/validation/test splits.")

group_ids = sorted(record["group_id"] for record in records)
random.Random(SEED).shuffle(group_ids)
n_test = max(1, round(len(group_ids) * 0.10))
n_validation = max(1, round(len(group_ids) * 0.10))
n_train = len(group_ids) - n_validation - n_test
if n_train < 1:
    raise ValueError("Not enough make/model groups to create all three splits.")

group_splits = {
    "train": set(group_ids[:n_train]),
    "validation": set(group_ids[n_train:n_train + n_validation]),
    "test": set(group_ids[n_train + n_validation:]),
}
records_by_group = {record["group_id"]: record for record in records}
records_by_split = {
    split: [records_by_group[group] for group in sorted(groups)]
    for split, groups in group_splits.items()
}
print("Unique make/model groups:", len(records))
print("Group counts:", {name: len(groups) for name, groups in group_splits.items()})


In [ ]:
PRIORITY_FIELDS = {
    "family": [
        ("seating capacity", "Seating_Capacity"),
        ("body style", "Body_Style"),
        ("price", "Formatted_Price"),
    ],
    "city": [
        ("city mileage", "Mileage_City"),
        ("fuel type", "Fuel_Type"),
        ("transmission", "Transmission"),
    ],
    "highway": [
        ("highway mileage", "Mileage_Highway"),
        ("power", "Power_BHP"),
        ("transmission", "Transmission"),
    ],
    "economy": [
        ("price", "Formatted_Price"),
        ("city mileage", "Mileage_City"),
        ("highway mileage", "Mileage_Highway"),
        ("fuel type", "Fuel_Type"),
    ],
    "performance": [
        ("power", "Power_BHP"),
        ("engine displacement", "Displacement"),
        ("transmission", "Transmission"),
    ],
    "offroad": [
        ("drivetrain", "Drivetrain"),
        ("body style", "Body_Style"),
    ],
}
PRIORITY_NAMES = {
    "family": "Family use",
    "city": "City driving",
    "highway": "Highway use",
    "economy": "Fuel economy and running costs",
    "performance": "Performance",
    "offroad": "Off-road use",
}

def display_fact(value):
    return value if value else "Not available"

examples_by_split = {name: [] for name in group_splits}
for split_name, split_records in records_by_split.items():
    for first, second in combinations(split_records, 2):
        for priority, fields in PRIORITY_FIELDS.items():
            comparisons = []
            first_facts = []
            second_facts = []
            for label, column in fields:
                value1 = first["values"].get(column, "")
                value2 = second["values"].get(column, "")
                if value1 or value2:
                    shown1 = display_fact(value1)
                    shown2 = display_fact(value2)
                    comparisons.append(f"{label}: {shown1} vs {shown2}")
                    first_facts.append(f"{label}: {shown1}")
                    second_facts.append(f"{label}: {shown2}")
            if comparisons:
                grounded_values = "; ".join(comparisons)
                first_values = "; ".join(first_facts)
                second_values = "; ".join(second_facts)
                target = (
                    f"For {PRIORITY_NAMES[priority]}, the dataset lists "
                    f"{grounded_values}. These listed fields support a specification "
                    "comparison but do not establish an overall buying recommendation."
                )
            else:
                grounded_values = "No comparable fields for this priority are available in these records."
                first_values = second_values = grounded_values
                target = (
                    f"For {PRIORITY_NAMES[priority]}, {grounded_values} "
                    "No vehicle specification should be inferred."
                )

            prompt = (
                f"Compare {first['name']} with {second['name']}.\n"
                f"Priority: {PRIORITY_NAMES[priority]}.\n"
                f"Vehicle 1 listed values: {first_values}\n"
                f"Vehicle 2 listed values: {second_values}\n"
                "Write a concise comparison using only the listed dataset values. "
                "Keep unavailable values unavailable."
            )
            examples_by_split[split_name].append({
                "pair_id": f"{first['group_id']} <> {second['group_id']}",
                "make_model_1": first["group_id"],
                "make_model_2": second["group_id"],
                "priority": priority,
                "prompt": prompt,
                "target": target,
            })

split_groups = {name: set(groups) for name, groups in group_splits.items()}
for left, right in combinations(split_groups, 2):
    overlap = split_groups[left] & split_groups[right]
    if overlap:
        raise RuntimeError(f"Make/model leakage between {left} and {right}: {sorted(overlap)[:5]}")

pair_sets = {
    name: {row["pair_id"] for row in examples}
    for name, examples in examples_by_split.items()
}
for left, right in combinations(pair_sets, 2):
    if pair_sets[left] & pair_sets[right]:
        raise RuntimeError(f"Vehicle-pair leakage between {left} and {right}.")

print("Actual comparison text examples:", {name: len(rows) for name, rows in examples_by_split.items()})
print("Total unique examples:", sum(len(rows) for rows in examples_by_split.values()))
print("Make/model overlap across splits: 0")
print("Vehicle-pair overlap across splits: 0")
print("Example prompt:", examples_by_split["train"][0]["prompt"])
print("Example target:", examples_by_split["train"][0]["target"])

split_dir = ARTIFACT_DIR / "grouped_splits"
split_dir.mkdir(parents=True, exist_ok=True)
for split_name, rows in examples_by_split.items():
    pd.DataFrame(rows).to_csv(split_dir / f"{split_name}.csv", index=False)


## 4. Shared character tokenizer and text streams

One character vocabulary is fitted from training text only and then reused for validation, test, and both models. Characters unseen in training map to one unknown token. The train, validation, and test streams remain separate so a training window cannot cross into held-out text.


In [ ]:
BLOCK_SIZE = 768
UNK_CHAR = "\ufffd"

def serialize_example(example):
    return f"{example['prompt']}\nAnswer: {example['target']}"

train_text = "\n\n".join(serialize_example(row) for row in examples_by_split["train"])
validation_text = "\n\n".join(serialize_example(row) for row in examples_by_split["validation"])
test_text = "\n\n".join(serialize_example(row) for row in examples_by_split["test"])

characters = sorted(set(train_text) - {UNK_CHAR})
stoi = {character: index for index, character in enumerate(characters)}
UNK_ID = len(stoi)
stoi[UNK_CHAR] = UNK_ID
itos = {index: character for character, index in stoi.items()}
itos[UNK_ID] = ""

def encode(text):
    return [stoi.get(character, UNK_ID) for character in text]

def decode(token_ids):
    return "".join(itos.get(int(token_id), "") for token_id in token_ids)

streams = {
    "train": torch.tensor(encode(train_text), dtype=torch.long),
    "validation": torch.tensor(encode(validation_text), dtype=torch.long),
    "test": torch.tensor(encode(test_text), dtype=torch.long),
}
DATA_SIGNATURE = hashlib.sha256("\0".join([train_text, validation_text, test_text]).encode("utf-8")).hexdigest()
print("Data/split fingerprint:", DATA_SIGNATURE)

if any(len(stream) <= BLOCK_SIZE + 1 for stream in streams.values()):
    raise ValueError("A split is too small for the 768-character context. Add more distinct make/model groups or reduce BLOCK_SIZE.")

print("Shared character vocabulary:", len(stoi))
print("Character/token counts:", {name: len(stream) for name, stream in streams.items()})
print("Training examples:", len(examples_by_split["train"]))
print("Validation examples:", len(examples_by_split["validation"]))
print("Test examples:", len(examples_by_split["test"]))


## 5. Decoder-only Transformer used for both runs

The block and state-dictionary names mirror the existing AutoNex GPT so the small model checkpoint can be loaded by the backend. The causal mask is built per sequence instead of being saved once per attention head. The large configuration uses the same tokenizer, context, block design, and data splits, with width and depth scaled up.


In [ ]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        key = self.key(x).unsqueeze(1)
        query = self.query(x).unsqueeze(1)
        value = self.value(x).unsqueeze(1)
        output = F.scaled_dot_product_attention(
            query,
            key,
            value,
            is_causal=True,
            dropout_p=self.dropout.p if self.training else 0.0,
        )
        return output.squeeze(1)


class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, n_embd, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(head_size, n_embd, dropout) for _ in range(n_head)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        output = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(output))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class AutoNexGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size=768,
        n_embd=128,
        n_head=4,
        n_layer=3,
        dropout=0.1,
        gradient_checkpointing=False,
    ):
        super().__init__()
        self.block_size = block_size
        self.gradient_checkpointing = gradient_checkpointing
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        batch_size, sequence_length = idx.shape
        if sequence_length > self.block_size:
            raise ValueError("Input sequence is longer than the model context.")
        positions = torch.arange(sequence_length, device=idx.device)
        x = self.token_embedding_table(idx) + self.position_embedding_table(positions)
        for block in self.blocks:
            if self.training and self.gradient_checkpointing:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(batch_size * sequence_length, -1),
                targets.reshape(batch_size * sequence_length),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=160, temperature=0.8):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(float(temperature), 0.01)
            next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx


def trainable_parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

SMALL_CONFIG = {
    "block_size": BLOCK_SIZE,
    "n_embd": 128,
    "n_head": 4,
    "n_layer": 3,
    "dropout": 0.1,
}
LARGE_CONFIG = {
    "block_size": BLOCK_SIZE,
    "n_embd": 1024,
    "n_head": 16,
    "n_layer": 40,
    "dropout": 0.1,
}

def parameter_count_formula(config, vocab_size):
    dim = config["n_embd"]
    layers = config["n_layer"]
    block_size = config["block_size"]
    # Per block: Q/K/V weights + output projection + 4x feed-forward + two LayerNorms.
    per_block = 12 * dim * dim + 10 * dim
    embeddings_and_head = 2 * vocab_size * dim + block_size * dim + 2 * dim + vocab_size
    return layers * per_block + embeddings_and_head

small_count_formula = parameter_count_formula(SMALL_CONFIG, len(stoi))
large_count_formula = parameter_count_formula(LARGE_CONFIG, len(stoi))
print(f"Small model exact formula count: {small_count_formula:,}")
print(f"Large model exact formula count: {large_count_formula:,} (~{large_count_formula / 1e6:.2f}M)")
print("The formula follows the declared embedding, attention, feed-forward, and normalization layers. Instantiated counts are checked against it when the runtime allows allocation.")


## 6. Check the active runtime before creating the 500M model

This estimate includes FP32 parameters, gradients, AdamW states, checkpointed activations, and a safety allowance. It is deliberately conservative. The actual available GPU and free memory are read now; the notebook does not assume a T4 or any other runtime.

A short smoke step estimates character tokens per second. The full large training cell remains separately disabled until you switch its flag on and review the estimated one-epoch time.


In [ ]:
def large_runtime_preflight(config, vocab_size, batch_size=1):
    parameter_count = parameter_count_formula(config, vocab_size)
    optimizer_and_weights = parameter_count * 20
    checkpointed_activations = (
        config["n_layer"] * batch_size * config["block_size"]
        * config["n_embd"] * 2 * 4
    )
    safety_bytes = 2 * 2**30
    estimated_peak = optimizer_and_weights + checkpointed_activations + safety_bytes

    report = {
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
        "parameter_count_estimate": parameter_count,
        "estimated_training_memory_gib": round(estimated_peak / 2**30, 2),
        "gpu_free_gib": None,
        "gpu_total_gib": None,
        "memory_gate_passed": False,
    }
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        report["gpu_free_gib"] = round(free_bytes / 2**30, 2)
        report["gpu_total_gib"] = round(total_bytes / 2**30, 2)
        report["memory_gate_passed"] = estimated_peak * 1.2 < free_bytes

    print(json.dumps(report, indent=2))
    if not torch.cuda.is_available():
        print("Large smoke/full training is skipped: an active CUDA GPU was not found.")
    elif not report["memory_gate_passed"]:
        print("Large smoke/full training is skipped: estimated memory plus headroom exceeds current free GPU memory.")
    return report

LARGE_PREFLIGHT = large_runtime_preflight(LARGE_CONFIG, len(stoi))


In [ ]:
def random_batch(stream, batch_size, block_size):
    maximum_start = len(stream) - block_size - 1
    starts = [random.randint(0, maximum_start) for _ in range(batch_size)]
    x = torch.stack([stream[start:start + block_size] for start in starts])
    y = torch.stack([stream[start + 1:start + block_size + 1] for start in starts])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss(model, eval_batches=20, batch_size=8):
    model.eval()
    results = {}
    for split_name in ("train", "validation", "test"):
        losses = []
        for _ in range(eval_batches):
            x, y = random_batch(streams[split_name], batch_size, model.block_size)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16,
                enabled=DEVICE.type == "cuda",
            ):
                _, loss = model(x, y)
            losses.append(float(loss.item()))
        results[split_name] = sum(losses) / len(losses)
    model.train()
    return results

def save_backend_checkpoint(path, model, config, metrics=None, step=0):
    backend_config = {
        "BLOCK_SIZE": config["block_size"],
        "N_EMBD": config["n_embd"],
        "N_HEAD": config["n_head"],
        "N_LAYER": config["n_layer"],
        "DROPOUT": config["dropout"],
    }
    payload = {
        "task": "dataset_grounded_vehicle_comparison",
        "model_state_dict": model.state_dict(),
        "vocab_size": len(stoi),
        "stoi": stoi,
        "itos": {str(index): token for index, token in itos.items()},
        "config": backend_config,
        "steps": step,
        "metrics": metrics or {},
    }
    torch.save(payload, path)

def save_resume_checkpoint(path, model, optimizer, config, step, history):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "data_signature": DATA_SIGNATURE,
        "step": step,
        "history": history,
        "stoi": stoi,
        "itos": itos,
        "torch_rng_state": torch.get_rng_state(),
        "python_rng_state": random.getstate(),
    }
    if torch.cuda.is_available():
        payload["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()
    temp_path = Path(str(path) + ".tmp")
    torch.save(payload, temp_path)
    temp_path.replace(path)

def run_training(
    model,
    config,
    steps,
    batch_size,
    learning_rate,
    grad_accum=1,
    eval_interval=250,
    eval_batches=10,
    save_interval=500,
    resume_path=None,
):
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    start_step = 0
    history = []

    if resume_path and Path(resume_path).is_file():
        resume = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        if resume.get("config") != config or resume.get("stoi") != stoi or resume.get("data_signature") != DATA_SIGNATURE:
            raise ValueError("Resume checkpoint configuration, tokenizer, or split data does not match this run.")
        model.load_state_dict(resume["model_state_dict"])
        optimizer.load_state_dict(resume["optimizer_state_dict"])
        start_step = int(resume.get("step", 0))
        history = resume.get("history", [])
        random.setstate(resume["python_rng_state"])
        torch.set_rng_state(resume["torch_rng_state"].cpu())
        if torch.cuda.is_available() and "cuda_rng_state_all" in resume:
            torch.cuda.set_rng_state_all(resume["cuda_rng_state_all"])
        print(f"Resumed at optimizer step {start_step}.")

    amp_dtype = (
        torch.bfloat16
        if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
        else torch.float16
    )
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)
    autocast_dtype = amp_dtype if use_amp else torch.float32
    model.train()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    character_tokens_seen = 0

    for step in range(start_step, steps):
        optimizer.zero_grad(set_to_none=True)
        for _ in range(grad_accum):
            x, y = random_batch(streams["train"], batch_size, model.block_size)
            character_tokens_seen += x.numel()
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=autocast_dtype,
                enabled=use_amp,
            ):
                _, loss = model(x, y)
                loss = loss / grad_accum
            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()
        if scaler.is_enabled():
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        actual_step = step + 1
        if eval_interval and actual_step % eval_interval == 0:
            losses = estimate_loss(model, eval_batches, batch_size)
            history.append({"step": actual_step, **losses})
            print(
                f"step {actual_step}: train {losses['train']:.4f}, "
                f"validation {losses['validation']:.4f}, test {losses['test']:.4f}"
            )
        if resume_path and save_interval and actual_step % save_interval == 0:
            save_resume_checkpoint(resume_path, model, optimizer, config, actual_step, history)

    elapsed = time.perf_counter() - started
    final_losses = estimate_loss(model, eval_batches, batch_size)
    peak_memory = (
        torch.cuda.max_memory_allocated() / 2**30
        if DEVICE.type == "cuda" else None
    )
    metrics = {
        "train_loss": final_losses["train"],
        "validation_loss": final_losses["validation"],
        "test_loss": final_losses["test"],
        "train_perplexity": math.exp(min(20, final_losses["train"])),
        "validation_perplexity": math.exp(min(20, final_losses["validation"])),
        "test_perplexity": math.exp(min(20, final_losses["test"])),
        "training_seconds": elapsed,
        "character_tokens_processed": character_tokens_seen,
        "character_tokens_per_second": character_tokens_seen / max(elapsed, 1e-9),
        "peak_gpu_memory_gib": peak_memory,
        "steps_completed_this_run": steps - start_step,
        "history": history,
    }
    return model, optimizer, metrics


## 7. Train the approximately 700K baseline from scratch

The small model uses the same 768-character context and decoder block used by the current backend model. Its exact count depends on the dataset-derived training vocabulary and is printed before training. The run starts with a fresh random initialization; it does not load the existing checkpoint.


In [ ]:
SMALL_STEPS = 3000
SMALL_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 4
SMALL_LEARNING_RATE = 0.0003
small_model = AutoNexGPT(len(stoi), **SMALL_CONFIG)
small_parameter_count = trainable_parameter_count(small_model)
if small_parameter_count != small_count_formula:
    raise RuntimeError("Small-model module count does not match the exact configuration formula.")
print(f"Actual small trainable parameters: {small_parameter_count:,}")

small_resume_path = ARTIFACT_DIR / "autonex_700k_resume.pt"
small_checkpoint_path = ARTIFACT_DIR / "autonex_decoder_700k_from_scratch.pt"
small_model, small_optimizer, small_metrics = run_training(
    small_model,
    SMALL_CONFIG,
    steps=SMALL_STEPS,
    batch_size=SMALL_BATCH_SIZE,
    learning_rate=SMALL_LEARNING_RATE,
    grad_accum=1,
    eval_interval=250,
    eval_batches=10,
    save_interval=500,
    resume_path=small_resume_path,
)
save_backend_checkpoint(
    small_checkpoint_path,
    small_model,
    SMALL_CONFIG,
    metrics=small_metrics,
    step=SMALL_STEPS,
)
small_config_path = ARTIFACT_DIR / "autonex_decoder_700k_config.json"
small_config_path.write_text(json.dumps({
    "model_name": "AutoNex decoder-only character GPT baseline",
    "trainable_parameters": small_parameter_count,
    "vocabulary_size": len(stoi),
    "config": SMALL_CONFIG,
    "metrics": {key: value for key, value in small_metrics.items() if key != "history"},
}, indent=2), encoding="utf-8")
small_vocab_path = ARTIFACT_DIR / "autonex_char_vocab.json"
small_vocab_path.write_text(json.dumps({
    "stoi": stoi,
    "itos": {str(index): token for index, token in itos.items()},
    "unknown_character": UNK_CHAR,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Small checkpoint:", small_checkpoint_path)
print("Small checkpoint size (MiB):", round(small_checkpoint_path.stat().st_size / 2**20, 2))


## 8. Large-model smoke test and optional full training

The next cell reads the active GPU's free memory and the estimate above before it allocates the large model. If the memory gate passes, it runs only three optimizer steps with gradient checkpointing and mixed precision where supported. That is a smoke test, not completed training.

Full 500M training is a separate opt-in. Its default is false. It estimates one approximate pass over the training stream from the measured smoke throughput and stops before training if that estimate exceeds the runtime budget you set.


In [ ]:
RUN_500M_SMOKE = True
large_smoke_metrics = None
large_smoke_seconds_per_character = None

if RUN_500M_SMOKE and LARGE_PREFLIGHT["memory_gate_passed"]:
    large_smoke_model = AutoNexGPT(
        len(stoi),
        **LARGE_CONFIG,
        gradient_checkpointing=True,
    )
    actual_large_count = trainable_parameter_count(large_smoke_model)
    if actual_large_count != large_count_formula:
        raise RuntimeError("Large-model module count does not match the exact configuration formula.")
    print(f"Actual large trainable parameters: {actual_large_count:,} ({actual_large_count / 1e6:.2f}M)")
    large_smoke_model, _, large_smoke_metrics = run_training(
        large_smoke_model,
        LARGE_CONFIG,
        steps=3,
        batch_size=1,
        learning_rate=0.00002,
        grad_accum=1,
        eval_interval=0,
        eval_batches=2,
        save_interval=0,
        resume_path=None,
    )
    large_smoke_seconds_per_character = (
        1.0 / max(large_smoke_metrics["character_tokens_per_second"], 1e-9)
    )
    print("Large-model smoke steps completed:", large_smoke_metrics["steps_completed_this_run"])
    print("Smoke throughput (character tokens/second):", round(large_smoke_metrics["character_tokens_per_second"], 2))
    print("Smoke peak GPU memory (GiB):", large_smoke_metrics["peak_gpu_memory_gib"])
    del large_smoke_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("500M smoke test skipped. The live runtime did not pass the memory gate or RUN_500M_SMOKE is false.")
    print("The parameter estimate is not evidence that training completed.")

RUN_500M_FULL = False
LARGE_RUNTIME_BUDGET_HOURS = 4.0
LARGE_BATCH_SIZE = 1
LARGE_GRAD_ACCUM = 8
LARGE_RESUME_PATH = ARTIFACT_DIR / "autonex_500m_resume.pt"
LARGE_CHECKPOINT_PATH = ARTIFACT_DIR / "autonex_decoder_500m_from_scratch.pt"

if RUN_500M_FULL:
    if not LARGE_PREFLIGHT["memory_gate_passed"] or large_smoke_metrics is None:
        raise RuntimeError("Full 500M training requires a passing active-runtime memory check and completed smoke steps.")
    large_steps_per_epoch = math.ceil(
        len(streams["train"]) / (LARGE_BATCH_SIZE * BLOCK_SIZE * LARGE_GRAD_ACCUM)
    )
    estimated_hours = (
        large_steps_per_epoch
        * LARGE_BATCH_SIZE
        * BLOCK_SIZE
        * LARGE_GRAD_ACCUM
        / max(large_smoke_metrics["character_tokens_per_second"], 1e-9)
        / 3600
    )
    print(f"Approximate one-pass step count: {large_steps_per_epoch:,}")
    print(f"Estimated one-pass time from smoke throughput: {estimated_hours:.2f} hours")
    if estimated_hours > LARGE_RUNTIME_BUDGET_HOURS:
        raise RuntimeError(
            "Estimated large-model training exceeds LARGE_RUNTIME_BUDGET_HOURS. "
            "Keep the full run off or raise the budget only after checking the active Colab session."
        )

    large_full_model = AutoNexGPT(
        len(stoi),
        **LARGE_CONFIG,
        gradient_checkpointing=True,
    )
    actual_large_count = trainable_parameter_count(large_full_model)
    if actual_large_count != large_count_formula:
        raise RuntimeError("Large-model module count does not match the exact configuration formula.")
    print(f"Actual large trainable parameters: {actual_large_count:,} ({actual_large_count / 1e6:.2f}M)")
    large_full_model, large_optimizer, large_metrics = run_training(
        large_full_model,
        LARGE_CONFIG,
        steps=large_steps_per_epoch,
        batch_size=LARGE_BATCH_SIZE,
        learning_rate=0.00002,
        grad_accum=LARGE_GRAD_ACCUM,
        eval_interval=100,
        eval_batches=4,
        save_interval=250,
        resume_path=LARGE_RESUME_PATH,
    )
    save_backend_checkpoint(
        LARGE_CHECKPOINT_PATH,
        large_full_model,
        LARGE_CONFIG,
        metrics=large_metrics,
        step=large_steps_per_epoch,
    )
    large_config_path = ARTIFACT_DIR / "autonex_decoder_500m_config.json"
    large_config_path.write_text(json.dumps({
        "model_name": "AutoNex decoder-only character GPT large experiment",
        "trainable_parameters": actual_large_count,
        "vocabulary_size": len(stoi),
        "config": LARGE_CONFIG,
        "metrics": {key: value for key, value in large_metrics.items() if key != "history"},
    }, indent=2), encoding="utf-8")
    print("Large checkpoint:", LARGE_CHECKPOINT_PATH)
    print("Large checkpoint size (GiB):", round(LARGE_CHECKPOINT_PATH.stat().st_size / 2**30, 2))
else:
    print("Full 500M training is OFF. Set RUN_500M_FULL = True only after reviewing hardware and time estimates.")


## 9. Inspect generated summaries for grounding and readability

The numeric screen below flags numbers in generated output that do not occur in the held-out prompt. It is only a basic warning signal: it cannot detect every unsupported qualitative claim. Read the examples yourself for factual consistency and whether the text reads well. Do not interpret this as an accuracy score.


In [ ]:
def generate_summary(model, example, max_new_tokens=220):
    prefix = example["prompt"] + "\nAnswer: "
    prompt_ids = torch.tensor([encode(prefix)], dtype=torch.long, device=DEVICE)
    generated = model.generate(prompt_ids, max_new_tokens=max_new_tokens, temperature=0.55)
    continuation = generated[0, prompt_ids.shape[1]:].tolist()
    text = decode(continuation).split("\n\n", 1)[0].strip()
    return text

def unsupported_numbers(text, source_prompt):
    generated_numbers = {
        token.replace(",", "")
        for token in re.findall(r"(?<!\w)\d+(?:[.,]\d+)?(?!\w)", text)
    }
    source_numbers = {
        token.replace(",", "")
        for token in re.findall(r"(?<!\w)\d+(?:[.,]\d+)?(?!\w)", source_prompt)
    }
    return sorted(generated_numbers - source_numbers)

heldout_examples = examples_by_split["test"][:5]
quality_rows = []
for example in heldout_examples:
    generated = generate_summary(small_model, example)
    unsupported = unsupported_numbers(generated, example["prompt"])
    quality_rows.append({
        "priority": example["priority"],
        "generated_summary": generated,
        "unsupported_numbers": unsupported,
        "human_readability_review_required": True,
    })
    print("\nPriority:", example["priority"])
    print("Generated:", generated)
    print("Unsupported numeric strings:", unsupported)

quality_path = ARTIFACT_DIR / "autonex_700k_heldout_samples.json"
quality_path.write_text(json.dumps(quality_rows, ensure_ascii=False, indent=2), encoding="utf-8")


In [ ]:
def checkpoint_size_mib(path):
    return round(Path(path).stat().st_size / 2**20, 2) if Path(path).is_file() else None

comparison_rows = [{
    "model": "About 700K from scratch",
    "trainable_parameters": small_parameter_count,
    "train_loss": small_metrics["train_loss"],
    "validation_loss": small_metrics["validation_loss"],
    "test_loss": small_metrics["test_loss"],
    "train_perplexity": small_metrics["train_perplexity"],
    "validation_perplexity": small_metrics["validation_perplexity"],
    "test_perplexity": small_metrics["test_perplexity"],
    "training_seconds": round(small_metrics["training_seconds"], 2),
    "character_tokens_per_second": round(small_metrics["character_tokens_per_second"], 2),
    "peak_gpu_memory_gib": small_metrics["peak_gpu_memory_gib"],
    "checkpoint_size_mib": checkpoint_size_mib(small_checkpoint_path),
}]
if RUN_500M_FULL and "large_metrics" in globals():
    comparison_rows.append({
        "model": "About 500M from scratch",
        "trainable_parameters": actual_large_count,
        "train_loss": large_metrics["train_loss"],
        "validation_loss": large_metrics["validation_loss"],
        "test_loss": large_metrics["test_loss"],
        "train_perplexity": large_metrics["train_perplexity"],
        "validation_perplexity": large_metrics["validation_perplexity"],
        "test_perplexity": large_metrics["test_perplexity"],
        "training_seconds": round(large_metrics["training_seconds"], 2),
        "character_tokens_per_second": round(large_metrics["character_tokens_per_second"], 2),
        "peak_gpu_memory_gib": large_metrics["peak_gpu_memory_gib"],
        "checkpoint_size_mib": checkpoint_size_mib(LARGE_CHECKPOINT_PATH),
    })
else:
    print("No full 500M metrics are reported because the optional full run was not completed.")

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
comparison_table.to_csv(ARTIFACT_DIR / "autonex_model_comparison_metrics.csv", index=False)


## 10. Save, download, and connect the small checkpoint

The small checkpoint is written separately from existing AutoNex files. Its model-state names and metadata match the backend loader. It can be placed in backend/model_files and selected with AUTONEX_SLM_CHECKPOINT. Do not replace the current checkpoint until you have reviewed the held-out examples and metrics.

The 500M model is not a practical default for the current CPU-oriented backend. Its checkpoint and configuration are kept separate. The model artifacts are saved under the artifact folder; use Google Drive for the large resumable file rather than a browser download.


In [ ]:
print("Small model files:")
for path in sorted(ARTIFACT_DIR.glob("autonex_*")):
    if path.is_file():
        print(f" - {path.name}: {path.stat().st_size / 2**20:.2f} MiB")

DOWNLOAD_SMALL_CHECKPOINT = False
if IS_COLAB and DOWNLOAD_SMALL_CHECKPOINT:
    files.download(str(small_checkpoint_path))
    files.download(str(small_config_path))
    files.download(str(small_vocab_path))

print(
    "To connect the selected small checkpoint locally, copy it to "
    "backend/model_files/autonex_tiny_gpt_700k_from_scratch.pt, then start "
    "the backend with AUTONEX_SLM_CHECKPOINT=autonex_tiny_gpt_700k_from_scratch.pt."
)
print("Keep the current autonex_tiny_gpt_checkpoint.pt as a separate rollback copy.")
